# 1024-Dimension Embedding Models Analysis

This notebook demonstrates the usage of high-dimensional embedding models (1024+ dimensions) from HuggingFace with automatic model downloading and caching. We'll analyze the codebase using the most effective embedding models available in our system.

## Overview

Our embedding system:
- ✅ **Automatic Download**: Downloads models from HuggingFace if not cached locally
- ✅ **Local Caching**: Stores models in `emmodels/` directory for faster subsequent loads
- ✅ **1024-Dimension Models**: Uses high-quality models for better semantic understanding
- ✅ **Fallback Mechanisms**: Handles errors gracefully with proper fallbacks

## Target Models (1024 Dimensions)
1. **BAAI/bge-large-en-v1.5** - 🥇 Best overall performance on MTEB benchmark
2. **thenlper/gte-large** - 🥈 Excellent speed/accuracy balance
3. **intfloat/e5-large-v2** - ⚡ Fast & reliable performance
4. **intfloat/multilingual-e5-large** - 🌍 Best multilingual support

## 1. Install Required Libraries

First, let's ensure all required libraries are installed for embedding generation and analysis.

In [ ]:
# Install required packages if not already installed
import subprocess
import sys

def install_package(package):
    """Install a package using pip"""
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ Successfully installed {package}")
    except subprocess.CalledProcessError as e:
        print(f"❌ Failed to install {package}: {e}")

# Required packages for our embedding analysis
required_packages = [
    "sentence-transformers",
    "torch",
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "plotly",
    "tqdm"
]

print("Installing required packages...")
for package in required_packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✅ {package} already installed")
    except ImportError:
        print(f"⬇️ Installing {package}...")
        install_package(package)

## 2. Import Dependencies

Import all necessary libraries and set up the environment for embedding analysis.

In [ ]:
# Core libraries
import os
import sys
import time
import logging
import warnings
from pathlib import Path
from typing import List, Dict, Tuple, Optional

# Data processing
import numpy as np
import pandas as pd

# Machine learning
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Progress tracking
from tqdm.auto import tqdm

# Add project root to path for importing our embedding system
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Suppress warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

print("✅ All dependencies imported successfully!")
print(f"📁 Working directory: {project_root}")
print(f"🐍 Python version: {sys.version}")

# Set plot style
plt.style.use('default')
sns.set_palette("husl")

## 3. Load Our Embedding System

Let's import and use our existing embedding system to leverage the sophisticated caching and model management.

In [ ]:
try:
    # Import our sophisticated embedding system
    from embeddings.providers import SentenceTransformerProvider, EmbeddingProviderFactory
    from embeddings.config import EMBEDDING_CONFIGS, EmbeddingConfig
    from embeddings.manager import EmbeddingManager
    
    print("✅ Successfully imported our embedding system!")
    print("🎯 Available embedding system features:")
    print("   - Automatic model downloading from HuggingFace")
    print("   - Local model caching in emmodels/ directory")
    print("   - Device compatibility fixes")
    print("   - Fallback mechanisms")
    
except ImportError as e:
    print(f"⚠️ Could not import embedding system: {e}")
    print("📝 Will use direct SentenceTransformer approach")

# Check if emmodels directory exists
emmodels_dir = project_root / "emmodels"
print(f"\n📁 Model cache directory: {emmodels_dir}")
print(f"   Exists: {'✅ Yes' if emmodels_dir.exists() else '❌ No (will be created)'}")

if emmodels_dir.exists():
    cached_models = list(emmodels_dir.iterdir())
    print(f"   Cached models: {len(cached_models)}")
    for model_dir in cached_models[:5]:  # Show first 5
        if model_dir.is_dir():
            print(f"   - {model_dir.name}")
    if len(cached_models) > 5:
        print(f"   ... and {len(cached_models) - 5} more")

## 4. Define High-Dimension Embedding Models (1024+)

Let's define the most effective 1024-dimension models available in our system and check their availability.

In [ ]:
# Define our target 1024-dimension models
TARGET_MODELS = {
    "bge-large-en": {
        "model_name": "BAAI/bge-large-en-v1.5",
        "dimensions": 1024,
        "description": "🥇 Best overall - Top MTEB performance, fast inference",
        "trust_remote_code": True,
        "use_case": "General purpose, English text, high accuracy"
    },
    "gte-large": {
        "model_name": "thenlper/gte-large", 
        "dimensions": 1024,
        "description": "🥈 Excellent speed - Very fast, great accuracy, multilingual",
        "trust_remote_code": False,
        "use_case": "Fast inference, good balance speed/accuracy"
    },
    "e5-large-v2": {
        "model_name": "intfloat/e5-large-v2",
        "dimensions": 1024, 
        "description": "⚡ Fast & reliable - Excellent speed/accuracy balance",
        "trust_remote_code": False,
        "use_case": "Production ready, stable, reliable"
    },
    "multilingual-e5-large": {
        "model_name": "intfloat/multilingual-e5-large",
        "dimensions": 1024,
        "description": "🌍 Multilingual - Best for 100+ languages",
        "trust_remote_code": False,
        "use_case": "Multi-language support, global applications"
    },
    "bge-m3": {
        "model_name": "BAAI/bge-m3",
        "dimensions": 1024,
        "description": "🔥 Multi-functionality - Multi-lingual, multi-granularity",
        "trust_remote_code": True,
        "use_case": "Advanced features, multiple languages"
    }
}

print("🎯 Target 1024-Dimension Models:")
print("=" * 60)
for key, model_info in TARGET_MODELS.items():
    print(f"\n📊 {key.upper()}")
    print(f"   Model: {model_info['model_name']}")
    print(f"   Dimensions: {model_info['dimensions']}")
    print(f"   Description: {model_info['description']}")
    print(f"   Use Case: {model_info['use_case']}")
    print(f"   Trust Remote Code: {model_info['trust_remote_code']}")

print(f"\n✅ Total models defined: {len(TARGET_MODELS)}")
print("📝 All models are 1024-dimensional for optimal semantic representation")